In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("zomato.csv")

df.head()
df.shape
df.info()
df.isnull().sum()

In [ ]:
#Remove Duplicate
df.duplicated().sum()

df = df.drop_duplicates(subset="res_id")
df.shape

In [ ]:
#Missing values
df.isnull().sum()

In [ ]:
#fill missing values
df["cuisines"] = df["cuisines"].fillna("Unknown")
df["timings"] = df["timings"].fillna("Not Available")
df["address"] = df["address"].fillna("Not Available")
df["opentable_support"] = df["opentable_support"].fillna(0)
df.drop(columns=["zipcode"], inplace=True)

In [ ]:
#Basic Statistics
df.describe()

In [ ]:
#Average restaurant rating:
df["aggregate_rating"].mean()


In [ ]:
#Rating Distribution
plt.figure(figsize=(8,5))
sns.histplot(df["aggregate_rating"], bins=20, kde=True)
plt.title("Distribution of Restaurant Ratings")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.show()

In [ ]:
#Rating Text Count
plt.figure(figsize=(8,5))
sns.countplot(data=df, x="rating_text", order=df["rating_text"].value_counts().index)
plt.xticks(rotation=45)
plt.title("Rating Text Distribution")
plt.show()

In [ ]:
                                                                       #Location Analysis
#City with Highest Restaurants
df["city"].value_counts().head(10)

In [ ]:
top_cities = df["city"].value_counts().head(10)

plt.figure(figsize=(10,6))
top_cities.plot(kind="barh")
plt.title("Top 10 Cities by Number of Restaurants")
plt.xlabel("Number of Restaurants")
plt.ylabel("City")
plt.show()

In [ ]:
#Average Rating by City
city_rating = df.groupby("city")["aggregate_rating"].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
city_rating.plot(kind="bar")
plt.title("Top Cities by Average Rating")
plt.ylabel("Average Rating")
plt.xticks(rotation=45)
plt.show()

In [ ]:
#Cuisine Analysis
from collections import Counter

cuisine_counter = Counter()

for cuisines in df["cuisines"]:
    for cuisine in str(cuisines).split(","):
        cuisine_counter[cuisine.strip()] += 1

top_cuisines = pd.DataFrame(cuisine_counter.most_common(10), columns=["Cuisine", "Count"])
top_cuisines

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=top_cuisines, x="Count", y="Cuisine")
plt.title("Top 10 Popular Cuisines")
plt.show()

In [ ]:
#Cuisine Variety vs Rating
df["cuisine_count"] = df["cuisines"].apply(lambda x: len(str(x).split(",")))

sns.scatterplot(data=df, x="cuisine_count", y="aggregate_rating")
plt.title("Cuisine Variety vs Rating")
plt.show()

df[["cuisine_count", "aggregate_rating"]].corr()

**Insight**: Restaurants offering more cuisine variety usually have slightly better ratings.

In [ ]:
#Price Range and Rating
price_rating = df.groupby("price_range").agg(
    restaurant_count=("res_id", "count"),
    avg_rating=("aggregate_rating", "mean"),
    avg_cost=("average_cost_for_two", "mean")
)

price_rating

In [ ]:
sns.barplot(data=df, x="price_range", y="aggregate_rating")
plt.title("Price Range vs Average Rating")
plt.show()

In [ ]:
#Average Cost for Two
sns.boxplot(data=df, x="price_range", y="average_cost_for_two")
plt.title("Average Cost for Two by Price Range")
plt.show()

In [ ]:
#Online Delivery Analysis
delivery_rating = df.groupby("delivery").agg(
    count=("res_id", "count"),
    avg_rating=("aggregate_rating", "mean")
)

delivery_rating

In [ ]:
sns.barplot(data=df, x="delivery", y="aggregate_rating")
plt.title("Delivery Availability vs Rating")
plt.show()

In [ ]:
#Top Restaurant Chains
top_chains = df["name"].value_counts().head(10)
top_chains

In [ ]:
plt.figure(figsize=(10,6))
top_chains.plot(kind="barh")
plt.title("Top Restaurant Chains by Outlets")
plt.xlabel("Number of Outlets")
plt.show()

In [ ]:
#Ratings of Top Chains
top_chain_names = top_chains.index

df[df["name"].isin(top_chain_names)].groupby("name")["aggregate_rating"].mean().sort_values(ascending=False)

In [ ]:
#Restaurant Features
df["has_wifi"] = df["highlights"].str.contains("Wifi", case=False, na=False)
df["has_alcohol"] = df["highlights"].str.contains("Alcohol", case=False, na=False)
df["has_outdoor"] = df["highlights"].str.contains("Outdoor Seating", case=False, na=False)
df["has_indoor"] = df["highlights"].str.contains("Indoor Seating", case=False, na=False)

In [ ]:
features = ["has_wifi", "has_alcohol", "has_outdoor", "has_indoor"]

for feature in features:
    print(df.groupby(feature)["aggregate_rating"].mean())

In [ ]:
#Scatter Plot
sns.scatterplot(data=df, x="average_cost_for_two", y="aggregate_rating")
plt.title("Average Cost vs Rating")
plt.show()

In [ ]:
#Box Plot
sns.boxplot(data=df, x="price_range", y="aggregate_rating")
plt.title("Rating by Price Range")
plt.show()

In [ ]:
#Density Plot
sns.kdeplot(df["aggregate_rating"], fill=True)
plt.title("Density Plot of Ratings")
plt.show()

In [ ]:
#Word Cloud
from wordcloud import WordCloud

text = " ".join(df["highlights"].astype(str))

wordcloud = WordCloud(width=1000, height=500, background_color="white").generate(text)

plt.figure(figsize=(12,6))
plt.imshow(wordcloud)
plt.axis("off")
plt.title("Word Cloud of Restaurant Features")
plt.show()

In [ ]:
#Correlation Heatmap
plt.figure(figsize=(8,6))
sns.heatmap(df[["average_cost_for_two", "price_range", "aggregate_rating", "votes", "photo_count"]].corr(),
            annot=True,
            cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

###**Final Insights**
- Bangalore has the highest number of restaurants.
- North Indian, Chinese, and Fast Food are the most popular cuisines.
- Average restaurant rating is around 2.96.
- Higher price range restaurants usually have higher ratings.
- Delivery availability improves average rating.
- Restaurants with Wi-Fi and alcohol availability have better ratings.
- Barbeque Nation has the highest rating among top restaurant chains.
- Average cost and rating have a positive relationship.
- Cuisine variety has a small positive effect on rating.
- Rating distribution shows many restaurants are average or good.